In [1]:
from ase import Atoms
from ase.data.colors import jmol_colors, cpk_colors
from vpython import vector
import vpython as vp
import numpy as np

# removes the canvas by default and allows to create a new
# scene when a new MolView is created
vp.scene.delete()

class AtomsView:
    def __init__(self, atoms: Atoms,
                 color_scheme: dict = jmol_colors,
                 show_axis: bool = False,
                 **kwargs) -> None:
        # next line imports the canvas already
        self.atoms = atoms
        self.scene = vp.canvas(**kwargs)
        self.axis = None
        self.show_axis(show=show_axis)
        self.color_scheme = color_scheme
        self.visualatoms = []
        self.dofs = {}
        # Unfortunately, vpython has problems removing objects (check vpython.baseObj.delete,
        # then I just keep them invisible, in case to reload,
        # just change to visibles dofs
        self.removed_dofs = {}
        self.add_atoms()
        # vpython such that the user can use vpython commands as well
        self.master = vp
        

    def add_atoms(self, atoms: Atoms = None,
                  color_scheme: np.ndarray=jmol_colors,
                  radius: float = 0.5):
        if atoms is None:
            atoms = self.atoms
        for atom in atoms:
            sphere = vp.sphere(pos=vector(*atom.position),
                               color=vector(*color_scheme[atom.number]),
                               canvas=self.scene,
                               radius=radius)
            self.visualatoms.append(sphere)

    def add_bond(self, atom1index, atom2index,
                 color=None, radius=0.2):
        """Add a bond between two atoms:
        atom1 and atom2

        Parameters
        ==========
        atom1index (and atom2index): int
            Indexes of the atoms to be connected according with g09
            convention.

        color: list. Default gray([0.5, 0.5, 0.5])
            RGB triplet.

        radius: float. Default 0.1
            Radius of the bond.

        Output
        ======

        Return the bonds in the system
        
        """
        # TODO: trajectories not yet implemented
        #if self.is_trajectory:
        #    atoms = self.atoms[self.viewer.view.frame]
        #else:
        atoms = self.atoms
        
        if color is None:
            color = [0.5, 0.5, 0.5]

        indexes = [atom1index, atom2index]
        if atom1index > atom2index:
            indexes = indexes[::-1]
        name = ''.join(str(i).zfill(3) for i in indexes)
        if name in self.dofs.keys():
            self.update_obj(self.dofs[name],
                            radius=radius,
                            color=vector(*color))
            return self.dofs[name]
        elif name in self.removed_dofs.keys():
            self.dofs[name] = self.removed_dofs[name]
            del self.removed_dofs[name]
            self.update_obj(self.dofs[name],
                            visible=True,
                            radius=radius,
                            color=vector(*color))
            return self.dofs[name]
            
            

        bond_extreme = atoms[atom1index - 1].position
        bond_axis = atoms[atom2index - 1].position - \
                    atoms[atom1index - 1].position

        b = vp.cylinder(pos=vector(*bond_extreme),
                        axis=vector(*bond_axis),
                        color=vector(*color),
                        radius=radius)

        self.dofs[name] = b

        return self.dofs[name]
    
    def remove_bond(self, atom1index, atom2index):
        """Remove a bond between two atoms:
        atoms1 and atoms2.

        Parameters
        ==========

        atom1index (and atom2index): int
            Indexes of the atoms that are connected. This bond
            will be removed.

        Output
        ======

        Return the bonds in the system
        """
        indexes = [atom1index, atom2index]
        indexes.sort()
        name = ''.join(str(i).zfill(3) for i in indexes)
        if name in self.dofs.keys():
            self.dofs[name].visible = False
            self.removed_dofs[name] = self.dofs[name]
            del self.dofs[name]
        return self.dofs
    
    def update_obj(self, obj, **kwargs):
        for attr, val in kwargs.items():
            if isinstance(val, (np.ndarray,
                                list,
                                tuple)):
                val = vector(*val)
            setattr(obj, attr, val)
        return obj
    
    def setting_canvas(self, **kwargs):
        scene = self.update_obj(self.scene, **kwargs)
        return scene
    
    def show_axis(self, show: bool = False,
                  size: float = 1.0):
        if self.axis is None:
            x = vp.arrow(pos=vector(0,0,0),
                         axis=vector(size,0,0),
                         color=vector(1,0,0),
                         canvas=self.scene)
            y = vp.arrow(pos=vector(0,0,0),
                         axis=vector(0,size,0),
                         color=vector(0,1,0),
                         canvas=self.scene)
            z = vp.arrow(pos=vector(0,0,0),
                         axis=vector(0,0,size),
                         color=vector(0,0,1),
                         canvas=self.scene)
            self.axis = [x, y, z]
        for ax in self.axis:
            ax.visualize = show
        
        return self.axis
        

    def add_triangle(self, v1, v2, v3, **kwargs):
        a = vp.vertex(pos=vector(*v1))
        b = vp.vertex(pos=vector(*v2))
        c = vp.vertex(pos=vector(*v3))
        tri = vp.triangle(v0=a, v1=b, v2=c)
        return tri

    def add_angle(self, atom1index, atom2index, atom3index,
                  color=None, n=0):
        """Add an angle to between three atoms:
        atom1, atom2 and atom3
        - with the vertex in the atom2

        Parameters
        ==========

        atom1index, atom2index and atom3index: int
            Indexes of the three atoms that defines the angle.
        color: color list. Default all gray([0.5, 0.5, 0.5])
            RGB triplet.
        n: int. Default 10
            number of intermedia points to add in the arc of
            the angle.

        Output
        ======
        Return the angles in the system
        """
        if color is None:
            color = [0.5, 0.5, 0.5]

        if self.is_trajectory:
            atoms = self.atoms[self.viewer.view.frame]
        else:
            atoms = self.atoms

        indexes = [atom1index, atom2index, atom3index]
        if atom1index > atom3index:
            indexes = indexes[::-1]
        name = ''.join(str(i).zfill(3) for i in indexes)
        self.remove_angle(atom1index, atom2index, atom3index)
        self.angles[name] = []

        vertex = atoms[atom2index - 1].position
        side1 = atoms[atom1index - 1].position - vertex
        side2 = atoms[atom3index - 1].position - vertex
        lenside1 = np.linalg.norm(side1)
        lenside2 = np.linalg.norm(side2)
        lensides = min(lenside1, lenside2)
        side1 = 0.7 * lensides * side1 / lenside1
        side2 = 0.7 * lensides * side2 / lenside2

        arcdots = [side1, side2]
        color = color * 3

        new = self.intermedia_vectors(side1,
                                      side2,
                                      n)

        if n != 0:
            [arcdots.insert(1, vert) for vert in new[::-1]]

        self.angles[name] = self.add_arc(vertex, arcdots, color)
        self.all_dofs_parameters[name] = (atom1index, atom2index, atom3index)

        return self.angles[name]

    def intermedia_vectors(self, a, b, n):
        """Define the intermedia arc dots between two vectors

        Parameters
        ==========

        a, b: arrays
             side vectors of the angles.
        n: int
             number of intermedia dots.

        Output
        ======
        Return the intermedia vectors between two side vectors.
        """

        if n == 0:
            return []
        n += 1
        c = b - a
        lena = np.linalg.norm(a)
        lenb = np.linalg.norm(b)
        lenc = np.linalg.norm(c)
        lend = min(lena, lenb)

        theta_total = np.arccos(np.dot(a, b) / (lena * lenb))

        
        beta = np.arccos(np.dot(a, c) / (lena * lenc))
        intermedia = []

        for i in range(1, n):
            theta = i * theta_total / n
            gamma = beta - theta
            factor = (lena * np.sin(theta)) / (lenc * np.sin(gamma))
            dird = a + factor * c
            d = lend * dird / np.linalg.norm(dird)
            intermedia.append(d)
        return intermedia

        
        


<IPython.core.display.Javascript object>

In [2]:
vp

<module 'vpython' from '/home/mbm/anaconda3/envs/sith_bc/lib/python3.9/site-packages/vpython/__init__.py'>

In [ ]:


class VisualAngle:
    def __init__(self, a, b, n, origin=None, scene=None, color=None,
                factor: float = 1):
        # number of intermedia vectors
        if color is None:
            self.color = vp.vector(0.5, 0.5, 0.5)
        else:
            self.color = self._asvector(color)
        self.n = n + 1
        self.factor = factor
        self.vertexes = [vp.vertex(pos=vp.vector(0,0,0),
                                   color=self.color) for _ in range(self.n)]

        a = self._asvector(a)
        b = self._asvector(b)
        if origin is None:
            origin = vp.vector(0, 0, 0)

        self.origin =  vp.vertex(pos=self._asvector(origin),
                                 color=self.color)
        self.update_vertexes(self.origin.pos, a, b)

        self.triangles = []
        for i in range(self.n - 1):
            self.triangles.append(vp.triangle(v0=self.origin,
                                              v1=self.vertexes[i],
                                              v2=self.vertexes[i + 1],
                                              color=self.color,
                                             canvas=scene))
    def _asvector(self, arraylike):
        if not isinstance(arraylike, vp.vector):
            arraylike = vp.vector(*arraylike)
        return arraylike

    def update_vertexes(self, origin=None, a=None, b=None, color=None):
        if a is None:
            a = self.vertexes[0].pos
        if b is None:
            b = self.vertexes[-1].pos
        if origin is not None:
            self.origin.pos = self._asvector(origin)
        if color is not None:
            color = self._asvector(color)
            self.origin.color = color
            for vertex in self.vertexes:
                vertex.color = color

        lena = a.mag
        lenb = b.mag
        axis = vp.cross(a, b).hat # normal vector
        if axis.mag == 0:
            axis = vp.cross(a, vp.vector(0,1,0))
        lend = self.factor * min(lena, lenb) # minumim length

        theta_total = vp.diff_angle(a, b)

        for i in range(self.n):
            theta = i * theta_total / (self.n - 1)
            self.vertexes[i].pos = self.origin.pos + vp.rotate(lend*a.hat, angle=theta, axis=axis)
        return self.vertexes
    
    def transport(self, v0):
        v0 = self._asvector(v0)
        self.origin.pos += v0
        for vertex in self.vertexes:
            vertex.pos += v0

    def rotate(self, angle, axis):
        #TODO : to be implemented
        return True
    
    def hide(self):
        for tri in self.triangles:
            tri.visible = False
        return self
    
    def show(self):
        for tri in self.triangles:
            tri.visible = True
        return self
        


In [ ]:
initialplace = np.array([[-1,-1,1],
                         [-0.5,0,0],
                         [0.5,0,0],
                         [1,1,1]])

In [ ]:
import SITH.SITH as s
s

In [1]:
!pip install nglview

  Using cached nglview-3.0.8-py3-none-any.whl


In [2]:
import nglview as nv
nv.show_ase(dumb_atoms)

In [7]:
nv.show_ase(dumb_atoms)

Exception ignored in: <function standardAttributes.__del__ at 0x7f4e5e8c00d0>
Traceback (most recent call last):
  File "/home/mbm/anaconda3/envs/sith_bc/lib/python3.9/site-packages/vpython/vpython.py", line 1159, in __del__
    super(standardAttributes, self).__del__()
  File "/home/mbm/anaconda3/envs/sith_bc/lib/python3.9/site-packages/vpython/vpython.py", line 340, in __del__
    cmd = {"cmd": "delete", "idx": self.idx}
AttributeError: 'arrow' object has no attribute 'idx'


NGLWidget()

In [2]:
pip install jupyterlab-vpython

Note: you may need to restart the kernel to use updated packages.


In [1]:
from SITH.visualize.view import VMolecule
from ase import Atoms
import numpy as np

dumb_atoms = Atoms('HOOH', 2*np.random.rand(4, 3))
#view = VMolecule(dumb_atoms, width=500)
#view.setting_canvas(background=(1,1,1))
view2 = VMolecule(dumb_atoms)
view2.setting_canvas(background=(1,0,1), width=100)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from vpython import *
from IPython.display import display
from ipywidgets import HBox

# Create the first VPython canvas
canvas1 = canvas(width=400, height=400)
sphere()

# Create the second VPython canvas
canvas2 = canvas(width=400, height=400)
box()

# Display the canvases side by side using HBox
HBox(children=[canvas1, canvas2])


In [ ]:
side1 = vp.cylinder(canvas=a.scene,
            pos=vp.vector(0, 0, 0),
            axis=vp.vector(0,1,1),
            radius=0.01)

side2 = vp.cylinder(canvas=a.scene,
            pos=vp.vector(0, 0, 0),
            axis=vp.vector(0,-1,1),
            radius=0.01)

In [ ]:
a = vp.vertex( pos=view.vatoms[0].pos, color=vp.vector(0,0,1) )
b = vp.vertex( pos=view.vatoms[1].pos, color=vp.vector(0,0,1) )
c = vp.vertex( pos=view.vatoms[2].pos, color=vp.vector(0,0,1) )
T = vp.triangle( v0=a, v1=b, v2=c )

In [ ]:
a1 = vp.vertex( pos=view.vatoms[3].pos, color=vp.vector(0,1,0) )
b1 = vp.vertex( pos=view.vatoms[1].pos, color=vp.vector(0,1,0) )
c1 = vp.vertex( pos=view.vatoms[2].pos, color=vp.vector(0,1,0) )
T1 = vp.triangle( v0=a1, v1=b1, v2=c1 )

In [ ]:
side1.radius = 0.01

In [ ]:
a.add_dihedral(1,2,3,4, color=(0.7,0.5,0))

In [ ]:
a.remove_bond(2,3)

In [ ]:
a.add_bond(2,3)

In [ ]:
a.add_angle(1,2,3)

In [ ]:
a.atoms[0].position = np.array([0,0,0])

In [ ]:
tri = VisualAngle([0,1,0], [-2,2,0], 100, scene=a.scene, color=(1,0,0))

In [ ]:
tri.transport([-1,-1,0])

In [ ]:
tri.triangles[0].v1.rotate(np.pi/2, axis=vector(0,0,1), origin=vector(-1,-1,0))

In [ ]:
tri.vertexes[0].rotate(np.pi/2)

In [ ]:
tri.transport([-1,-1,0])

In [ ]:
tri.rotate()

In [ ]:
tri = VisualAngle([2,0,0], [2,2,0], 5, scene=a.scene, color=(1,0,0))

In [ ]:
tri.origin.pos=vector(0,1,0)

In [ ]:
tri.update_vertexes(a=vector(0,2,0),
                    b=vector(-2,0,0), origin=vector(1,2,3))

In [ ]:
tri.triangles[0].v1.color = vector(0,0,0)

In [ ]:
A = 

In [ ]:
A

In [ ]:
A.x = 0.5

In [ ]:
a = vp.vertex( pos=vector(0,0,0) )
b = vp.vertex( pos=vp.vec(1,0,0) )
c = vp.vertex( pos=vp.vec(1,1,0) )
T = vp.triangle( v0=a, v1=b, v2=c , color=vp.vector(1,0,0))

In [ ]:
T.color = vector(1,0,0)

In [ ]:
T.v0.pos = vector(0,1,0)

In [ ]:
tri.origin.pos = vector(0,0,0)

In [ ]:
A

In [ ]:
tri.transport([1,1,1])

In [ ]:
tri.triangles[-3].visible=False

In [ ]:
tri.origin = vector(1,2,3)

In [ ]:
tri.update_vertexes(vector(1,0,0), vector(0,1,0))

In [ ]:
tri.triangles[0].v0.pos
.0

In [ ]:
b = vp.cylinder(pos=vector(-2,0,0),
                axis=vector(2,0,0),
                color=vector(1, 0.5, 0.1),
                radius=0.5,
                canvas=None)
                

In [ ]:
t = a.add_triangle([0, 0, 0],
               [2, 0, 0],
               [0, 2, 0])

In [ ]:
t.visible = True

In [ ]:
v0 = vertex( pos=vector(-1,-1,0) )

In [ ]:
v0.pos = vector(0,0,0)

In [ ]:
t.v0.pos = vector(2,2,0)

In [ ]:
vars(t.v0)

In [ ]:
t.color=vector(1,1,1)

In [ ]:
t.pos = vector(-1,0,0)

In [ ]:
t.rotate(angle=np.pi/4,
         axis=vector(0,0,1),
         origin=vector(0,0,0))

In [ ]:
a = vertex( pos=vec(0,0,0) )
b = vertex( pos=vec(1,0,0) )
c = vertex( pos=vec(1,1,0) )
T = triangle( v0=a, v1=b, v2=c )



In [ ]:
a.scene.background = vector(1,1,1)

In [ ]:
a.add_axis()

In [ ]:
s.visible = False

In [ ]:
s = extrusion(path=[vector(0, 0, 3), vector(0,0,3.5)], 
              shape=[shapes.circle(radius=1),
                     shapes.circle(radius=1,
                                  angle1=0,
                                  angle2=np.pi/4)])

In [ ]:
def move_tri():
    s.pos += vector(0.1,0,0)

In [ ]:
scene.bind('click', move_tri)

In [ ]:
parts[0].axis

In [ ]:
s = extrusion(path=[vector(0, 0, 3), vector(0,0,3.5)], 
              shape=shapes.circle(radius=1,
                                  angle1=0,
                                  angle2=np.pi/4))

In [ ]:
parts[1].

In [ ]:
parts[9].visible = False

In [ ]:
p = 10
angle = 2*np.pi/p

parts = []
for i in range(p):
    s = extrusion(path=[vector(0, 0, 3), vector(0,0,3.5)], 
                  shape=shapes.circle(radius=1,
                                      angle1=i*angle,
                                      angle2=(i+1)*angle))
    parts.append(s)
    

In [ ]:
cir.visible = False

In [ ]:
for part in parts:
    part.visible = True

In [ ]:
cir.pos = vector(1.2,0,0)

In [ ]:
cir = compound(parts)

In [ ]:
s._path.append([0,0,0])


In [ ]:
import numpy as np

In [ ]:
vector(0,1,0)

In [ ]:
res.updates

In [ ]:
res = compound([T2, T])

In [ ]:
T2 = triangle(v0=vertex( pos=vec(0.5,0,0) ),
              v1=vertex( pos=vec(0.4,1,0) ),
              v2=vertex( pos=vec(0,1,0) ) )

In [ ]:
T = triangle(v0=vertex( pos=vec(0,0,0) ),
             v1=vertex( pos=vec(1,0,0) ),
             v2=vertex( pos=vec(1,1,0) ) )

In [ ]:
s.angle2=np.pi/2

In [ ]:
s = extrusion(path=[vector(0, 0, 3), vector(0,0,3.5)], 
              shape=shapes.circle(radius=1,
                                  angle1=0,
                                  angle2=np.pi/4))

In [ ]:
s.angle

In [ ]:
vars(a.scene)

In [ ]:
a.add_bond(1, 2)
a.add_bond(3, 2)
a.add_bond(3, 1)


In [ ]:
a.remove_bond(1, 2)

In [ ]:
a.bonds['001002'].visible = False

In [ ]:
a.bonds

In [ ]:
a.bonds['001002'].delete()

In [ ]:
dumb_atoms[0].position

In [ ]:
jmol_colors[dumb_atoms.numbers]

In [ ]:
from ase import Atoms

In [ ]:
dumb_atoms.numbers

In [ ]:
import numpy as np

vp.vector(*vp.vector(1,2,3))

In [ ]:
def dumb(**args):
    print('cinco' in args.keys())

In [ ]:
dumb(cuatro=4)

In [ ]:
vp.scene.delete()

In [ ]:
import vpython as vp
from ase.data.colors import jmol_colors, cpk_colors

In [ ]:
scene3.updates['cmd'](cmd)

In [ ]:
scene3 = vp.canvas()

In [ ]:
vp.sphere(canvas=scene2)

In [ ]:
vp.scene.delete()

In [ ]:
vars(vp.scene)

In [ ]:
vp.scene.background = vp.vector(1,1,1)

In [ ]:
setattr(vp.scene, 'background', vp.vector(0,0,0))

In [ ]:
from vpython import 
sphere()

In [ ]:
from delete import show_sphere

In [ ]:
show_sphere()

In [ ]:
vp.canvas()